# 04 — Results

Loads the persisted RF model + metrics, plots the confusion matrix and ROC curve, and (once the partner's SVM is ready) renders a side-by-side comparison table.

The SVM block at the bottom is gated on a `svm_metrics.json` file — leave it commented out until the partner drops their metrics into `data/processed/`.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

from bme_ml.paths import setup_paths
from bme_ml.splits import Splits, select_rows, add_subject_id
from bme_ml.models import Xy
from bme_ml.features import FEATURE_NAMES
from bme_ml.evaluation import BinaryMetrics, compare

paths = setup_paths()
rf = joblib.load(paths.models / 'rf_binary.joblib')
splits = Splits.from_json(paths.splits_json)
rf_metrics = BinaryMetrics(**json.loads((paths.processed / 'rf_metrics.json').read_text()))

In [ ]:
features = add_subject_id(pd.read_parquet(paths.features_parquet))
test_df = select_rows(features, splits.test_subjects)
X_test, y_test = Xy(test_df)
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Normal', 'Abnormal'], ax=axes[0])
axes[0].set_title('Random Forest — Confusion')
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title('Random Forest — ROC')
plt.tight_layout(); plt.show()

In [ ]:
# Side-by-side comparison. CNN row appears once 05_train_cnn has run.
# Uncomment the SVM block once your partner drops svm_metrics.json into data/processed/.
rows = [('Random Forest', rf_metrics)]

cnn_metrics_path = paths.processed / 'cnn_metrics.json'
if cnn_metrics_path.exists():
    cnn_metrics = BinaryMetrics(**json.loads(cnn_metrics_path.read_text()))
    rows.append(('1D CNN', cnn_metrics))

# svm_metrics_path = paths.processed / 'svm_metrics.json'
# if svm_metrics_path.exists():
#     svm_metrics = BinaryMetrics(**json.loads(svm_metrics_path.read_text()))
#     rows.append(('SVM (partner)', svm_metrics))

compare(rows)